In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import confusion_matrix, roc_auc_score, accuracy_score

# --- 1. Wczytanie przygotowanego zbioru danych ---
cleaned_data_path = os.path.join('f1_data_cleaned', 'f1_processed_dataset.csv')
df_f1 = pd.read_csv(cleaned_data_path)

# --- 2. Definicja predyktorów i zmiennej celu ---
categorical_features = ['driver_nationality', 'constructor_nationality']
numerical_features = ['grid', 'year', 'round', 'circuitId', 'driver_age', 'quali_position']

X = df_f1[categorical_features + numerical_features]
y = df_f1['top3']

# --- 3. Podział na zbiór uczący i testowy (seed=1, proporcja 70/30) ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1, stratify=y
)

# --- 4. Transformacja zmiennych kategorycznych przy użyciu ColumnTransformer ---
# DODANO: handle_unknown='ignore', aby obsłużyć rzadkie kategorie typu 'East German'
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    remainder='passthrough'
)

X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)

# Pobranie nazw cech po transformacji dla celów wizualizacji drzewa
encoded_cat_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features).tolist()
all_feature_names = encoded_cat_names + numerical_features

print(f"Dane gotowe do modelowania!")
print(f"Liczba cech po transformacji One-Hot: {X_train_trans.shape[1]}")

In [ ]:
# Wariant A: Drzewo pełne (brak ograniczeń struktury)
tree_wariant_a = DecisionTreeClassifier(criterion='gini', random_state=1)
tree_wariant_a.fit(X_train_trans, y_train)

# Wariant B: Drzewo optymalne (Pre-pruning: głębokość i min liść)
tree_wariant_b = DecisionTreeClassifier(criterion='gini', max_depth=4, min_samples_leaf=20, random_state=1)
tree_wariant_b.fit(X_train_trans, y_train)

# Wariant C: Drzewo zbalansowane (Pre-pruning + wagi klas)
tree_wariant_c = DecisionTreeClassifier(criterion='gini', max_depth=4, min_samples_leaf=20, class_weight='balanced', random_state=1)
tree_wariant_c.fit(X_train_trans, y_train)
print("Modele zostały pomyślnie wytrenowane!")

In [ ]:
modele_drzewiaste = {
    'Wariant A (Pełne)': tree_wariant_a,
    'Wariant B (Optymalne)': tree_wariant_b,
    'Wariant C (Zbalansowane)': tree_wariant_c
}

podsumowanie_wynikow = []

for nazwa, model in modele_drzewiaste.items():
    pred_etykiety = model.predict(X_test_trans)
    pred_prawdopodobienstwa = model.predict_proba(X_test_trans)[:, 1]

    tn, fp, fn, tp = confusion_matrix(y_test, pred_etykiety).ravel()

    dokladnosc = accuracy_score(y_test, pred_etykiety)
    czulosc = tp / (tp + fn)      # Sensitivity
    specyficznosc = tn / (tn + fp)  # Specificity
    auc = roc_auc_score(y_test, pred_prawdopodobienstwa)

    podsumowanie_wynikow.append({
        'Model': nazwa,
        'Accuracy': round(dokladnosc, 4),
        'Czułość (Sensitivity)': round(czulosc, 4),
        'Specyficzność (Specificity)': round(specyficznosc, 4),
        'AUC Score': round(auc, 4)
    })

df_podsumowanie = pd.DataFrame(podsumowanie_wynikow)
display(df_podsumowanie)

In [ ]:
plt.figure(figsize=(24, 12))
plot_tree(tree_wariant_b, 
          feature_names=all_feature_names, 
          class_names=['Brak Podium', 'Podium'], 
          filled=True, 
          rounded=True, 
          fontsize=10)
plt.title("Graficzna struktura podziałów dla optymalnego drzewa decyzyjnego (Wariant B)", fontsize=16)
plt.show()

print("\n--- REGUŁY DECYZYJNE GENEROWANE PRZEZ DRZEWO (WARIANT B) ---")
print(export_text(tree_wariant_b, feature_names=all_feature_names))